In [1]:
from pyspark.sql import SparkSession # type: ignore

spark = SparkSession.builder \
    .appName("SparkCourse") \
    .master("local[*]") \
    .config("spark.sql.warehouse.dir", "/home/jovyan/work/setup/spark-warehouse") \
    .config("spark.hadoop.javax.jdo.option.ConnectionURL",
            "jdbc:derby:/home/jovyan/work/metastore_db;create=true") \
    .config("spark.hadoop.javax.jdo.option.ConnectionDriverName",
            "org.apache.derby.jdbc.EmbeddedDriver") \
    .enableHiveSupport() \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 3.5.0


In [2]:
spark.sql("show databases").show()
spark.sql("show tables in spark_db").show()

+---------+
|namespace|
+---------+
|  default|
| spark_db|
+---------+

+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
| spark_db|            bookings|      false|
| spark_db|          facilities|      false|
| spark_db|             members|      false|
| spark_db|    offline_students|      false|
| spark_db|offline_students_raw|      false|
| spark_db|     online_students|      false|
+---------+--------------------+-----------+



In [3]:
print("Member table:")
spark.sql("SELECT * FROM spark_db.members").show()

print("Facilities table:")
spark.sql("SELECT * FROM spark_db.facilities").show()

print("Bookings table:")
spark.sql("SELECT * FROM spark_db.bookings").show()

Member table:


+-----+---------+---------+--------------------+-------+--------------+-------------+-------------------+
|memid|  surname|firstname|             address|zipcode|     telephone|recommendedby|           joindate|
+-----+---------+---------+--------------------+-------+--------------+-------------+-------------------+
|    0|    GUEST|    GUEST|               GUEST|      0|(000) 000-0000|         NULL|2022-07-01 00:00:00|
|    1|    Smith|   Darren|8 Bloomsbury Clos...|   4321|  555-555-5555|         NULL|2022-07-02 12:02:05|
|    2|    Smith|    Tracy|8 Bloomsbury Clos...|   4321|  555-555-5555|         NULL|2022-07-02 12:08:23|
|    3|   Rownam|      Tim|23 Highway Way, B...|  23423|(844) 693-0723|         NULL|2022-07-03 09:32:15|
|    4| Joplette|   Janice|20 Crossing Road,...|    234|(833) 942-4710|            1|2022-07-03 10:25:05|
|    5|  Butters|   Gerald|1065 Huntingdon A...|  56754|(844) 078-4130|            1|2022-07-09 10:44:09|
|    6|    Tracy|   Burton|3 Tunisia Drive, ..

In [4]:
"""
List all bookings made by a person named Darren Smith as the following.

member_id | first_name | last_name | address | facility_id | slots
---------------------------------------------------------------------
Ensure the following

    Show the details of all persons named Darren Smith even if they have not made any bookings
    Sort the result by number of slots (highers first)
    List the person with no bookings at the top
"""
from pyspark.sql.functions import col # type: ignore

members_df = spark.table("spark_db.members").alias("m")
bookings_df = spark.table("spark_db.bookings").alias("b")

members_bookings_join = col("m.memid") == col("b.memid")

result_df = members_df.join(bookings_df, members_bookings_join, "left")\
                    .filter((col("m.firstname") == "Darren") & 
                            (col("m.surname") == "Smith"))\
                    .select(
                        col("m.memid").alias("member_id"),
                        col("m.firstname").alias("first_name"),
                        col("m.surname").alias("last_name"),
                        col("m.address"),
                        col("b.facid").alias("facility_id"),
                        col("b.slots")
                    )\
                    .orderBy(col("b.slots").desc_nulls_first()) # desc_nulls_first() function is used to sort but with NULLs on top # desc_nulls_last() is another function that does the opposite
result_df.show()

+---------+----------+---------+--------------------+-----------+-----+
|member_id|first_name|last_name|             address|facility_id|slots|
+---------+----------+---------+--------------------+-----------+-----+
|       37|    Darren|    Smith|3 Funktown, Denzi...|       NULL| NULL|
|        1|    Darren|    Smith|8 Bloomsbury Clos...|          2|    9|
|        1|    Darren|    Smith|8 Bloomsbury Clos...|          2|    6|
|        1|    Darren|    Smith|8 Bloomsbury Clos...|          2|    6|
|        1|    Darren|    Smith|8 Bloomsbury Clos...|          2|    6|
|        1|    Darren|    Smith|8 Bloomsbury Clos...|          2|    6|
|        1|    Darren|    Smith|8 Bloomsbury Clos...|          2|    6|
|        1|    Darren|    Smith|8 Bloomsbury Clos...|          2|    6|
|        1|    Darren|    Smith|8 Bloomsbury Clos...|          2|    6|
|        1|    Darren|    Smith|8 Bloomsbury Clos...|          2|    6|
|        1|    Darren|    Smith|8 Bloomsbury Clos...|          2

In [5]:
"""
Show me a bookings report for Darren Smith as the following.

facility_name | slots | booking_amount | start_time | member_id | member_name | telephone | address
------------------------------------------------------------------------------------------------------
The report must meet the following criteria.

    Show the details of all persons named Darren Smith even if they have not made any bookings
    Sort the result by number of slots (highers first)
    List the person with no bookings at the top
"""
from pyspark.sql.functions import concat_ws # type: ignore

members_filtered_df = members_df.filter((col("m.firstname") == "Darren") &
                                        (col("m.surname") == "Smith"))\
                                .alias("m")
bookings_df = spark.table("spark_db.bookings").alias("b")
facilities_df = spark.table("spark_db.facilities").alias("f")

members_bookings_join = col("m.memid") == col("b.memid")
bookings_facilities_join = col("b.facid") == col("f.facid")

joined_df = members_filtered_df.join(bookings_df, members_bookings_join, "left")\
                            .join(facilities_df, bookings_facilities_join, "left")

result_df = joined_df.select(
    col("f.fac_name").alias("facility_name"),
    col("b.slots"),
    (col("b.slots") * col("f.membercost")).alias("booking_amount"),
    col("b.starttime").alias("start_time"),
    col("m.memid").alias("member_id"),
    concat_ws("-", col("m.firstname"), col("m.surname")).alias("member_name"),
    col("m.telephone"),
    col("m.address")
).orderBy(col("b.slots").desc_nulls_first())

result_df.show()

+---------------+-----+--------------+-------------------+---------+------------+--------------+--------------------+
|  facility_name|slots|booking_amount|         start_time|member_id| member_name|     telephone|             address|
+---------------+-----+--------------+-------------------+---------+------------+--------------+--------------------+
|           NULL| NULL|          NULL|               NULL|       37|Darren-Smith|(822) 577-3541|3 Funktown, Denzi...|
|Badminton Court|    9|             0|2022-08-28 13:30:00|        1|Darren-Smith|  555-555-5555|8 Bloomsbury Clos...|
|Badminton Court|    6|             0|2022-09-30 14:00:00|        1|Darren-Smith|  555-555-5555|8 Bloomsbury Clos...|
|Badminton Court|    6|             0|2022-09-10 09:00:00|        1|Darren-Smith|  555-555-5555|8 Bloomsbury Clos...|
|Badminton Court|    6|             0|2022-09-09 13:00:00|        1|Darren-Smith|  555-555-5555|8 Bloomsbury Clos...|
|Badminton Court|    6|             0|2022-09-07 14:00:0

In [12]:
"""
Prepare a facility booking report as the following

facility_name | member_cost | gest_cost | start_time | slots
---------------------------------------------------------------
Ensure the following

    All club facilities must be listed in the report
    Consider only bookings for more than 10 slots
"""
bookings_filtered_df = bookings_df.filter(col("b.slots") > 10)

result_df = facilities_df.join(bookings_filtered_df, bookings_facilities_join, "left")\
                        .select(
                            facilities_df.fac_name.alias("facility_name"),
                            facilities_df.membercost.alias("member_cost"),
                            facilities_df.guestcost.alias("guest_cost"),
                            bookings_filtered_df.starttime,
                            bookings_filtered_df.slots
                        )
result_df.show()

+---------------+-----------+----------+-------------------+-----+
|  facility_name|member_cost|guest_cost|          starttime|slots|
+---------------+-----------+----------+-------------------+-----+
| Tennis Court 1|          5|        25|2022-09-15 08:00:00|   12|
| Tennis Court 2|          5|        25|               NULL| NULL|
|Badminton Court|          0|      NULL|               NULL| NULL|
|   Table Tennis|          0|         5|               NULL| NULL|
| Massage Room 1|         35|        80|               NULL| NULL|
| Massage Room 2|         35|        80|               NULL| NULL|
|   Squash Court|       NULL|      NULL|2022-09-13 10:30:00|   14|
|  Snooker Table|          0|         5|               NULL| NULL|
|     Pool Table|          0|         5|               NULL| NULL|
+---------------+-----------+----------+-------------------+-----+



In [21]:
"""
Prepare a member bookings report as the following

booking_id | facility_name | slots | first_name | last_name | address
Ensure the following

    Consider only regular memebrs (not guest) and direct members(not recomended by any other member)
    Consider only bookings for more than 8 hours
    Ensure all regular and direct members are listed even if they have no 8 hour bookings
    Ensure all 8 hour bookings are listed even if they are not made by regular and direct members
    Sort the report by slots and first name in ascending order
"""

regular_direct_members = members_df.filter((col("firstname") != "GUEST") &
                                           (col("recommendedby").isNull()))\
                                   .alias("rdm")
bookings_day_df = bookings_df.filter(col("slots") > 8).alias("bdf")

members_bookings_join = col("rdm.memid") == col("bdf.memid")
bookings_facilities_join = col("bdf.facid") == col("f.facid")
full_join_facilities_join = col("fjd.facid") == col("f.facid")

full_join_df = regular_direct_members.join(bookings_day_df, members_bookings_join, "full").alias("fjd")

result_df = full_join_df.join(facilities_df, full_join_facilities_join, "left")\
                        .select(
                            full_join_df.bookid.alias("booking_id"),
                            facilities_df.fac_name.alias("facility_name"),
                            full_join_df.slots,
                            full_join_df.firstname.alias("first_name"),
                            full_join_df.surname.alias("last_name"),
                            full_join_df.address.alias("address")
                        )\
                        .orderBy(col("fjd.slots").asc_nulls_last(), col("fjd.firstname").asc_nulls_last())
result_df.show()

+----------+---------------+-----+----------+----------+--------------------+
|booking_id|  facility_name|slots|first_name| last_name|             address|
+----------+---------------+-----+----------+----------+--------------------+
|      1927|Badminton Court|    9|    Darren|     Smith|8 Bloomsbury Clos...|
|      1757| Tennis Court 2|    9|      NULL|      NULL|                NULL|
|      3041| Tennis Court 1|    9|      NULL|      NULL|                NULL|
|      3563| Tennis Court 1|    9|      NULL|      NULL|                NULL|
|      3768| Tennis Court 2|    9|      NULL|      NULL|                NULL|
|      3836| Tennis Court 2|    9|      NULL|      NULL|                NULL|
|       530| Tennis Court 1|    9|      NULL|      NULL|                NULL|
|       660| Tennis Court 2|    9|      NULL|      NULL|                NULL|
|      2978| Tennis Court 1|   12|      NULL|      NULL|                NULL|
|      2888|   Squash Court|   14|      NULL|      NULL|        